In [ ]:
import numpy as np

import numpy.linalg as la

import matplotlib.pyplot as plt

import cv2

from PIL import Image

import pickle

import os

In [ ]:
DATA_DIR = './cifar-10-batches-py'

# Load a single CIFAR-10 batch from disk
def load_batch(filepath):

    with open(filepath, 'rb') as f:

        d = pickle.load(f, encoding='bytes')

    images = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

    labels = np.array(d[b'labels'])

    return images, labels

# Load human-readable class names from batches.meta
def load_class_names(data_dir):

    meta_filepath = os.path.join(data_dir, 'batches.meta')

    with open(meta_filepath, 'rb') as f:

        d = pickle.load(f, encoding='bytes')

    label_names = [name.decode('utf-8') for name in d[b'label_names']]

    return label_names

# Load all 5 training batches and concatenate
train_batches = [load_batch(os.path.join(DATA_DIR, f'data_batch_{i}')) for i in range(1, 6)]

train_images = np.concatenate([b[0] for b in train_batches])

train_labels = np.concatenate([b[1] for b in train_batches])

# Load test batch
test_images, test_labels = load_batch(os.path.join(DATA_DIR, 'test_batch'))

# Load class names
class_names = load_class_names(DATA_DIR)

print(train_images.shape, test_images.shape)

In [ ]:
# SIFT setup -- lowered thresholds to recover keypoints on small CIFAR images.
sift = cv2.SIFT_create(contrastThreshold=0.02, edgeThreshold=5)

# Upscale to 128x128 so the Gaussian scale space has enough room to detect features.
# Returns: float32 array of shape (N_keypoints, 128), or None if no keypoints found.
def extract_sift(img_rgb):

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    img_up = cv2.resize(gray, (128, 128))

    _, descs = sift.detectAndCompute(img_up, None)

    return descs

# Vocabulary building: sample 5000 training images, collect all SIFT descriptors,
# then run k-means to produce a K-word visual codebook.
K = 500

N_VOCAB_IMAGES = 5000

np.random.seed(42)

vocab_indices = np.random.choice(len(train_images), size=N_VOCAB_IMAGES, replace=False)

all_descs = []

for idx, i in enumerate(vocab_indices):

    d = extract_sift(train_images[i])

    if d is not None:

        all_descs.append(d)

    if idx % 1000 == 0:

        print(f'Vocabulary descriptors: {idx}/{N_VOCAB_IMAGES}')

all_descs = np.vstack(all_descs).astype(np.float32)

print(f'Total descriptors: {len(all_descs)}')

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)

_, _, codebook = cv2.kmeans(all_descs, K, None, criteria, 3, cv2.KMEANS_RANDOM_CENTERS)

codebook = codebook.astype(np.float32)

print(f'Codebook shape: {codebook.shape}')

# BoVW encoding: assign each SIFT descriptor to its nearest codebook word,
# build a frequency histogram over the K words, then L2-normalise.
# Nearest-neighbor assignment uses the squared-L2 identity to avoid a large
# (N_desc x K x 128) intermediate tensor.
def encode_bovw(descs, codebook, K):

    if descs is None or len(descs) == 0:

        return np.zeros(K, dtype=np.float32)

    descs = descs.astype(np.float32)

    a2 = np.sum(descs ** 2, axis=1, keepdims=True)

    b2 = np.sum(codebook ** 2, axis=1, keepdims=True).T

    ab = descs @ codebook.T

    assignments = np.argmin(a2 + b2 - 2 * ab, axis=1)

    hist = np.bincount(assignments, minlength=K).astype(np.float32)

    hist = hist / (la.norm(hist) + 1e-7)

    return hist

# Extract a 96-dim normalized color histogram from an RGB image
def extract_color_histogram(img_rgb):

    hist_r = np.histogram(img_rgb[:, :, 0], bins=32, range=(0, 256))[0]

    hist_g = np.histogram(img_rgb[:, :, 1], bins=32, range=(0, 256))[0]

    hist_b = np.histogram(img_rgb[:, :, 2], bins=32, range=(0, 256))[0]

    feat = np.concatenate([hist_r, hist_g, hist_b]).astype(np.float32)

    feat = feat / (feat.sum() + 1e-7)

    return feat

# Extract a 256-dim normalized LBP histogram from an RGB image
def extract_lbp(img_rgb):

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    offsets = [(-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1)]

    h, w = gray.shape

    center = gray[1:-1, 1:-1].astype(np.float32)

    lbp = np.zeros((h - 2, w - 2), dtype=np.uint8)

    for k_bit, (di, dj) in enumerate(offsets):

        neighbor = gray[1 + di:h - 1 + di, 1 + dj:w - 1 + dj].astype(np.float32)

        lbp |= ((neighbor >= center).astype(np.uint8) << k_bit)

    hist, _ = np.histogram(lbp.ravel(), bins=256, range=(0, 256))

    feat = hist.astype(np.float32)

    feat = feat / (feat.sum() + 1e-7)

    return feat

# Extract a 16-dim GLCM Haralick feature vector from an RGB image
def extract_glcm(img_rgb):

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    N = 8

    gray_q = (gray.astype(np.int32) * N // 256).clip(0, N - 1)

    h, w = gray_q.shape

    offsets = [(0, 1), (1, 0), (1, 1), (1, -1)]

    ii, jj = np.meshgrid(np.arange(N), np.arange(N), indexing='ij')

    feats = []

    for di, dj in offsets:

        r0 = max(0, -di)

        r1 = h + min(0, -di)

        c0 = max(0, -dj)

        c1 = w + min(0, -dj)

        ref = gray_q[r0:r1, c0:c1]

        neigh = gray_q[r0 + di:r1 + di, c0 + dj:c1 + dj]

        glcm = np.zeros((N, N), dtype=np.float32)

        np.add.at(glcm, (ref.ravel(), neigh.ravel()), 1)

        glcm = glcm + glcm.T

        glcm = glcm / (glcm.sum() + 1e-7)

        contrast = np.sum(glcm * (ii - jj) ** 2)

        homogeneity = np.sum(glcm / (1.0 + np.abs(ii - jj)))

        energy = np.sqrt(np.sum(glcm ** 2))

        mu_i = np.sum(ii * glcm)

        mu_j = np.sum(jj * glcm)

        sig_i = np.sqrt(np.sum(glcm * (ii - mu_i) ** 2) + 1e-7)

        sig_j = np.sqrt(np.sum(glcm * (jj - mu_j) ** 2) + 1e-7)

        correlation = np.sum(glcm * (ii - mu_i) * (jj - mu_j)) / (sig_i * sig_j)

        feats.extend([contrast, homogeneity, energy, correlation])

    feat = np.array(feats, dtype=np.float32)

    feat = feat / (la.norm(feat) + 1e-7)

    return feat

# Histogram of Canny Edge Responses
# Divide the Canny binary edge map into an n_cells x n_cells spatial grid.
# Record edge pixel density (fraction of edge pixels) per cell.
# Result: 16-dim L2-normalised spatial edge-density vector.
def extract_canny(img_rgb, n_cells=4, low=50, high=150):

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    edges = cv2.Canny(gray, low, high)

    h, w = edges.shape

    ch, cw = h // n_cells, w // n_cells

    feat = []

    for i in range(n_cells):

        for j in range(n_cells):

            feat.append(edges[i * ch:(i + 1) * ch, j * cw:(j + 1) * cw].mean())

    feat = np.array(feat, dtype=np.float32) / 255.0

    feat = feat / (la.norm(feat) + 1e-7)

    return feat

# Histogram of Sobel Edge Responses
# Compute per-pixel gradient magnitude via Sobel operators in x and y.
# Bin magnitudes into n_bins buckets over the full response range [0, 1500].
# Result: 32-dim L2-normalised magnitude-distribution histogram.
def extract_sobel(img_rgb, n_bins=32):

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)

    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    magnitude = np.sqrt(sobel_x ** 2 + sobel_y ** 2)

    hist, _ = np.histogram(magnitude.ravel(), bins=n_bins, range=(0.0, 1500.0))

    feat = hist.astype(np.float32)

    feat = feat / (la.norm(feat) + 1e-7)

    return feat

# Combined descriptor: L2-normalise each sub-descriptor, then concatenate.
# All 6 representations contribute equally regardless of their native normalisation scheme.
# Result: 916-dim vector (96 + 256 + 16 + 16 + 32 + 500).
def extract_combined(img_rgb, codebook, K):

    c = extract_color_histogram(img_rgb)

    l = extract_lbp(img_rgb)

    g = extract_glcm(img_rgb)

    ca = extract_canny(img_rgb)

    s = extract_sobel(img_rgb)

    b = encode_bovw(extract_sift(img_rgb), codebook, K)

    def norm(v):

        return v / (la.norm(v) + 1e-7)

    return np.concatenate([norm(c), norm(l), norm(g), norm(ca), norm(s), norm(b)]).astype(np.float32)

feat_test_combined = extract_combined(train_images[0], codebook, K)

assert feat_test_combined.shape == (916,)

assert feat_test_combined.dtype == np.float32

In [ ]:
k = 10

N_QUERIES = 100

TOTAL_RELEVANT = 5000

db_color = np.zeros((len(train_images), 96), dtype=np.float32)

db_lbp = np.zeros((len(train_images), 256), dtype=np.float32)

db_glcm = np.zeros((len(train_images), 16), dtype=np.float32)

db_canny = np.zeros((len(train_images), 16), dtype=np.float32)

db_sobel = np.zeros((len(train_images), 32), dtype=np.float32)

db_bovw = np.zeros((len(train_images), K), dtype=np.float32)

# Extract all 6 descriptors for every training image in a single pass
for i in range(len(train_images)):

    db_color[i] = extract_color_histogram(train_images[i])

    db_lbp[i] = extract_lbp(train_images[i])

    db_glcm[i] = extract_glcm(train_images[i])

    db_canny[i] = extract_canny(train_images[i])

    db_sobel[i] = extract_sobel(train_images[i])

    db_bovw[i] = encode_bovw(extract_sift(train_images[i]), codebook, K)

    if i % 10000 == 0:

        print(f'Processed {i}/50000')

assert db_color.shape == (50000, 96)

assert db_lbp.shape == (50000, 256)

assert db_glcm.shape == (50000, 16)

assert db_canny.shape == (50000, 16)

assert db_sobel.shape == (50000, 32)

assert db_bovw.shape == (50000, K)

print(db_color.shape, db_lbp.shape, db_glcm.shape, db_canny.shape, db_sobel.shape, db_bovw.shape)

# Build combined descriptor database: L2-normalise each component row-wise, then concatenate
def l2_norm_rows(X):

    norms = la.norm(X, axis=1, keepdims=True)

    return X / (norms + 1e-7)

db_combined = np.concatenate([

    l2_norm_rows(db_color),

    l2_norm_rows(db_lbp),

    l2_norm_rows(db_glcm),

    l2_norm_rows(db_canny),

    l2_norm_rows(db_sobel),

    l2_norm_rows(db_bovw),

], axis=1).astype(np.float32)

assert db_combined.shape == (50000, 916)

print(db_combined.shape, db_combined.dtype)

In [ ]:
# Retrieve top-k nearest neighbours by L2 distance
def retrieve(query_feat, db_features, k=10):

    dists = la.norm(db_features - query_feat, axis=1)

    top_k_idx = np.argsort(dists)[:k]

    return top_k_idx, dists[top_k_idx]

# Fraction of retrieved results matching the query label
def precision_at_k(query_label, retrieved_labels, k):

    count = np.sum(retrieved_labels[:k] == query_label)

    return count / k

# Fraction of all relevant items retrieved in top-k
def recall_at_k(query_label, retrieved_labels, k, total_relevant):

    count = np.sum(retrieved_labels[:k] == query_label)

    return count / total_relevant

# Reciprocal rank of the first correct result
def reciprocal_rank(query_label, retrieved_labels):

    for i, label in enumerate(retrieved_labels):

        if label == query_label:

            return 1.0 / (i + 1)

    return 0.0

# Display query image and its top-k retrieved neighbours
def show_retrieval(query_img, query_label, top_k_imgs, top_k_labels, top_k_dists, class_names, k=10):

    fig, axes = plt.subplots(1, k + 1, figsize=(2 * (k + 1), 3))

    axes[0].imshow(query_img)

    axes[0].set_title('Query: ' + class_names[query_label])

    axes[0].axis('off')

    for j in range(k):

        axes[j + 1].imshow(top_k_imgs[j])

        axes[j + 1].set_title(class_names[top_k_labels[j]] + '\n' + str(round(top_k_dists[j], 3)))

        axes[j + 1].axis('off')

    plt.tight_layout()

    plt.show()

# Smoke test retrieval with combined descriptor
query_idx = 0

query_vec = db_combined[0]

top_k_idx, top_k_dists = retrieve(query_vec, db_combined, k=10)

assert len(top_k_idx) == 10

assert top_k_dists[0] <= top_k_dists[-1]

print(top_k_idx, top_k_dists)

In [ ]:
np.random.seed(42)

query_indices = np.random.choice(len(test_images), size=N_QUERIES, replace=False)

descriptors = [

    ('Color Histogram', lambda img: extract_color_histogram(img), db_color),

    ('LBP', lambda img: extract_lbp(img), db_lbp),

    ('GLCM', lambda img: extract_glcm(img), db_glcm),

    ('Canny Edge', lambda img: extract_canny(img), db_canny),

    ('Sobel Edge', lambda img: extract_sobel(img), db_sobel),

    ('BoVW SIFT', lambda img: encode_bovw(extract_sift(img), codebook, K), db_bovw),

    ('Combined', lambda img: extract_combined(img, codebook, K), db_combined),

]

results = {}

for name, extractor, db_feat in descriptors:

    precisions = []

    recalls = []

    rr_scores = []

    for qi in query_indices:

        q_img = test_images[qi]

        q_label = test_labels[qi]

        q_feat = extractor(q_img)

        top_k_idx, _ = retrieve(q_feat, db_feat, k=k)

        retrieved_labels = train_labels[top_k_idx]

        precisions.append(precision_at_k(q_label, retrieved_labels, k))

        recalls.append(recall_at_k(q_label, retrieved_labels, k, TOTAL_RELEVANT))

        rr_scores.append(reciprocal_rank(q_label, retrieved_labels))

    results[name] = (np.mean(precisions), np.mean(recalls), np.mean(rr_scores))

    print(f'{name}: P@{k}={results[name][0]:.3f}  R@{k}={results[name][1]:.3f}  MRR={results[name][2]:.3f}')

print()

print(f'{"Descriptor":<22} | P@{k:<4} | R@{k:<4} | MRR')

print('-' * 22 + ' | ----- | ----- | -----')

for name, (p, r, mrr) in results.items():

    print(f'{name:<22} | {p:.3f} | {r:.3f} | {mrr:.3f}')

In [ ]:
# Demo retrieval on a random test image with the combined descriptor
sample_idx = np.random.randint(len(test_images))

query_img = test_images[sample_idx]

query_label = test_labels[sample_idx]

query_feat = extract_combined(query_img, codebook, K)

top_k_idx, top_k_dists = retrieve(query_feat, db_combined, k=k)

top_k_imgs = train_images[top_k_idx]

top_k_labels = train_labels[top_k_idx]

show_retrieval(query_img, query_label, top_k_imgs, top_k_labels, top_k_dists, class_names, k=k)